In [32]:
from dataclasses import dataclass, field
from typing import Callable, Optional
import math
import numpy as np 

@dataclass
class FunctionEvaluator:
    func: Callable[..., float]

    args: tuple = field(default_factory=tuple)
    kwargs: dict = field(default_factory=dict)


    def eval_at(self, x: float) -> float:
        return self.func(x, *self.args, **self.kwargs)



evaluator = FunctionEvaluator(
    func=lambda x, : 2 * x**2 + 3* x + 5)
print(evaluator.eval_at(x=2))



evaluator_2 = FunctionEvaluator(
    func=lambda x, a, b, c: a * x**2 + b * x + c,
    args=(2, 3, 5)
)
print(evaluator_2.eval_at(x=2))


evaluator_3 = FunctionEvaluator(
    func=lambda x: math.sin(x)
)
print(evaluator_3.eval_at(x=math.pi/2))

evaluator_4 = FunctionEvaluator(
    func=lambda x: math.exp(x)
)
print(evaluator_4.eval_at(x=1))

objects = [evaluator, evaluator_2, evaluator_3, evaluator_4]

for i in objects:
    print(i.eval_at(x=2))


19
19
1.0
2.718281828459045
19
19
0.9092974268256817
7.38905609893065


In [24]:
@dataclass
class EquationDistrubutedLoad:
    start_postion: float
    end_postion: float
    func: Callable[..., float]
    args: tuple = field(default_factory=tuple)
    kwargs: dict = field(default_factory=dict)


    def magnitude_at(self, position: float) -> float:
        if position < self.start_postion or position > self.end_postion:
            return 0.0
        else:
            # Evaluate the equation at the given position
            return self.func(position, *self.args, **self.kwargs)
            

load_1 = EquationDistrubutedLoad(
    start_postion=2,
    end_postion=5,
    func=lambda x: 2*x)

print(load_1.magnitude_at(position=10))

0.0


In [15]:
from dataclasses import dataclass, field
from typing import Callable, Optional

@dataclass
class DistrubutedLoad:
    # Start and end positions of the load
    start_position: float
    end_position: float

    # Function for the load
    func: Optional[Callable[..., float]] = None
    start_magnitude: Optional[float] = None
    end_magnitude: Optional[float] = None

    args: tuple = field(default_factory=tuple)
    kwargs: dict = field(default_factory=dict)

    

    def __post_init__(self):
            if self.func is not None:
                return

            if self.start_magnitude is  not None and self.end_magnitude is not None:
                if self.start_position == self.end_position:
                    raise ValueError("Start and stop cannot be the same value")

                if self.start_magnitude == self.end_magnitude:
                    self.func = lambda x: self.start_magnitude
                    return

                else:
                    y0, y1 = self.start_magnitude, self.end_magnitude
                    x0, x1 = self.start_position, self.end_position

                    self.func = lambda x: y0 + (x - x0) * (y1 - y0) / (x1 - x0)
                    return

            raise ValueError("Either func or start_magnitude and end_magnitude must be provided")

    def eval_at(self, position: float) -> float:
        if not(self.start_position <= position <= self.end_position):
             return 0.0
        
        return self.func(position, *self.args, **self.kwargs)

load_1 = DistrubutedLoad(
    start_magnitude=0,
    end_magnitude=8,
    start_position=0,
    end_position=5
)

print(load_1.eval_at(position=2.5))

load_2 = DistrubutedLoad(
    start_position=0,
    end_position=5,
    func=lambda x: 2*x
)

print(load_2.eval_at(position=2.5))

    

4.0
5.0
